# Read in and clean data

In [13]:
import pandas as pd
import numpy as np
from glob import glob
import nltk
from nltk.stem.porter import PorterStemmer

nltk_resources = [
    'punkt_tab',
    'averaged_perceptron_tagger_eng',
    'stopwords',
    'tagsets'
]
for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource)

# Load all CSVs
dfs = []
for f in sorted(glob('SongLyrics/csv/*.csv')):
    dfs.append(pd.read_csv(f))

df = pd.concat(dfs, ignore_index=True)

# Drop placeholder rows
before = len(df)
df = df[~df['Lyric'].str.contains("lyrics for this song have yet to be released", case=False, na=False)]
after = len(df)
print(f"Rows dropped: {before - after}")
print(f"Rows remaining: {after}")

# Drop duplicate songs
df = df.drop_duplicates(subset=['Artist', 'Title'])

# Remove BTS, NickiMinaj, Cardi B, Eminem, Drake (not in English/profanity)
df = df[df['Artist'] != 'BTS (방탄소년단)']
df = df[df['Artist'] != 'Nicki Minaj']
df = df[df['Artist'] != 'Cardi B']
df = df[df['Artist'] != 'Eminem']
df = df[df['Artist'] != 'Drake']

# Remove songs with non-English lyrics
df['Lyric'] = df['Lyric'].fillna('').str.replace(
    r'[^A-Za-z0-9\s.,!?\'"()\-\n]', '', regex=True
)

#Remove instrumental and interlude tracks with no meaningful lexical content following normalization
df = df[df['Lyric'].str.strip() != '']

# Reset index — this becomes song_id
df = df.dropna(subset=['Lyric'])
df = df.reset_index(drop=True)
df.index.name = 'song_id'

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/natalieseah/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/natalieseah/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/natalieseah/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package tagsets to
[nltk_data]     /Users/natalieseah/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!


Rows dropped: 227
Rows remaining: 5800


# Build LIB

In [14]:
LIB = df[['Artist', 'Title', 'Album', 'Date', 'Year']].copy()
LIB['lyric_len'] = df['Lyric'].str.len()
LIB

,Artist,Title,Album,Date,Year,lyric_len
song_id,,,,,,
0,Ariana Grande,"​thank u, next","thank u, next",2018-11-03,2018.0,2305
1,Ariana Grande,7 rings,"thank u, next",2019-01-18,2019.0,2184
2,Ariana Grande,​God is a woman,Sweetener,2018-07-13,2018.0,1975
3,Ariana Grande,Side To Side,Dangerous Woman,2016-05-20,2016.0,2725
4,Ariana Grande,​​no tears left to cry,Sweetener,2018-04-20,2018.0,2307
...,...,...,...,...,...,...
4110,Taylor Swift,Teardrops on my Guitar (Live from Clear Channe...,Live From Clear Channel Stripped 2008,2008-06-28,2008.0,1374
4111,Taylor Swift,Evermore [Forward],NaN,2020-12-11,2020.0,744
4112,Taylor Swift,Welcome Back Grunwald,NaN,NaN,NaN,317


In [15]:
print(LIB['lyric_len'].mean())

LIB.to_csv('/Users/natalieseah/Desktop/TextasData/SongLyrics/LIB.csv', sep='|')

1643.6371810449575


# Build CORPUS

In [16]:
OHCO = ['song_id', 'sent_num', 'token_num']

def tokenize_song(song_id):
    lyric = df.loc[song_id, 'Lyric']
    
    # Tokenize into sentences
    sentences = nltk.sent_tokenize(lyric)
    SENTS = pd.Series(sentences, name='sent_str').to_frame()
    SENTS.index.name = 'sent_num'
    
    # Tokenize into tokens with POS tags
    TOKENS = SENTS.sent_str.apply(
        lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x)))
    ).stack().to_frame('pos_tuple')
    
    TOKENS['token_str'] = TOKENS.pos_tuple.apply(lambda x: x[0])
    TOKENS['pos'] = TOKENS.pos_tuple.apply(lambda x: x[1])
    TOKENS['pos_group'] = TOKENS.pos.str[:2]
    TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r'[\W_]+', '', regex=True)
    TOKENS = TOKENS.drop('pos_tuple', axis=1)
    
    # Remove empty terms
    TOKENS = TOKENS[TOKENS.term_str != ''].copy()
    TOKENS.index.names = ['sent_num', 'token_num']
    
    return TOKENS

corpus = []
for song_id in df.index:
    print(f"Tokenizing: {df.loc[song_id, 'Title']}")
    corpus.append(tokenize_song(song_id))

CORPUS = pd.concat(corpus, keys=df.index)
CORPUS.index.names = OHCO
CORPUS


Tokenizing: ​thank u, next
Tokenizing: 7 rings
Tokenizing: ​God is a woman
Tokenizing: Side To Side
Tokenizing: ​​no tears left to cry
Tokenizing: ​​breathin
Tokenizing: ​break up with your girlfriend, i’m bored
Tokenizing: ​positions
Tokenizing: 34+35
Tokenizing: ​imagine
Tokenizing: ​needy
Tokenizing: ​pov
Tokenizing: ​ghostin
Tokenizing: ​pete davidson
Tokenizing: ​in my head
Tokenizing: ​sweetener
Tokenizing: ​R.E.M.
Tokenizing: NASA
Tokenizing: Dangerous Woman
Tokenizing: Let Me Love You
Tokenizing: ​bloodline
Tokenizing: Into You
Tokenizing: ​fake smile
Tokenizing: ​goodnight n go
Tokenizing: ​the light is coming
Tokenizing: One Last Time
Tokenizing: ​bad idea
Tokenizing: ​better off
Tokenizing: ​make up
Tokenizing: ​get well soon
Tokenizing: The Way
Tokenizing: Problem
Tokenizing: ​everytime
Tokenizing: Focus
Tokenizing: Everyday
Tokenizing: Moonlight
Tokenizing: ​just like magic
Tokenizing: ​raindrops (an angel cried)
Tokenizing: ​nasty
Tokenizing: ​successful
Tokenizing: Be Al

token_str  pos pos_group term_str
song_id sent_num token_num                                  
0       0        0           thought   NN        NN  thought
                 1                 i   NN        NN        i
                 2                'd   MD        MD        d
                 3               end   VB        VB      end
                 4                up   RP        RP       up
...                              ...  ...       ...      ...
4114    0        81                i   NN        NN        i
                 82               'm  VBP        VB        m
                 83             done  VBN        VB     done
                 84          finding   NN        NN  finding
                 85                u   NN        NN        u

[1466714 rows x 4 columns]

In [17]:
CORPUS.to_csv('/Users/natalieseah/Desktop/TextasData/SongLyrics/CORPUS.csv', sep='|')

# Build VOCAB

In [18]:
VOCAB = CORPUS.term_str.value_counts().to_frame('n').sort_index()
VOCAB.index.name = 'term_str'

# Probability and information content
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()
VOCAB['i'] = -np.log2(VOCAB.p)

# Max POS
VOCAB['max_pos'] = CORPUS[['term_str','pos']].value_counts().unstack(fill_value=0).idxmax(1)
VOCAB['max_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().unstack(fill_value=0).idxmax(1)

# Stopwords
sw = pd.DataFrame(nltk.corpus.stopwords.words('english'), columns=['term_str']).set_index('term_str')
sw['stop'] = 1
VOCAB['stop'] = sw['stop']
VOCAB['stop'] = VOCAB['stop'].fillna(0).astype(int)

# Custom stops
custom_stops = ['nt', 'ca', 'lyrics', 'snippet', 'unreleased', 'wan']
VOCAB.loc[VOCAB.index.isin(custom_stops), 'stop'] = 1

# Porter stem
stemmer = PorterStemmer()
VOCAB['porter_stem'] = VOCAB.index.map(lambda x: stemmer.stem(x))

# Document frequency and DFIDF
N = df.shape[0]
VOCAB['df'] = CORPUS.reset_index().groupby('term_str')['song_id'].nunique()
VOCAB['dfidf'] = VOCAB['df'] / N * VOCAB['i']

VOCAB

,n,p,i,max_pos,max_pos_group,stop,porter_stem,df,dfidf
term_str,,,,,,,,,
0,185,1.261323e-04,12.952775,CD,CD,0,0,105,0.330508
00,41,2.795364e-05,15.126604,CD,CD,0,00,32,0.117631
000,14,9.545146e-06,16.676801,CD,CD,0,000,6,0.024316
00000,1,6.817962e-07,20.484156,CD,CD,0,00000,1,0.004978
000s,1,6.817962e-07,20.484156,CD,CD,0,000,1,0.004978
...,...,...,...,...,...,...,...,...,...
zuzu,1,6.817962e-07,20.484156,NN,NN,0,zuzu,1,0.004978
zwrotka,2,1.363592e-06,19.484156,NN,NN,0,zwrotka,1,0.004735
zwycizc,1,6.817962e-07,20.484156,NN,NN,0,zwycizc,1,0.004978


In [19]:
# Number of documents (songs)
N = df.shape[0]

# Document frequency — how many songs each term appears in
DF = CORPUS.reset_index().groupby('term_str')['song_id'].nunique().rename('df')

# Add to VOCAB
VOCAB['df'] = DF
VOCAB['dfidf'] = VOCAB['df'] / N * VOCAB['i']

# Top 20 significant words by DFIDF
print(VOCAB[VOCAB.stop == 0].sort_values('dfidf', ascending=False).head(20).index.tolist())

['know', 'like', 'cause', 'got', 'pre', 'love', 'oh', 'go', 'na', 'time', 'get', 'see', 'say', 'let', 'never', 'one', 'baby', 'yeah', 'make', 'take']


In [20]:
VOCAB.to_csv('/Users/natalieseah/Desktop/TextasData/VOCAB.csv', sep='|')